# H infinity condition for stability of a discrete time system

In [5]:
import numpy as np
import cvxpy as cp
import control as ct

Matrizes do sistema

In [6]:
A  = np.array([
    [1, 0.035],
    [0, -0.13]
])
Bu = np.array([
    [-0.002],
    [-0.11]
])
Bw = np.array([
    [0.1],
    [0.1]
])
C  = np.array([
    [1, 0]
])
# C = np.eye(A.shape[0])
Du = np.zeros(shape=(C.shape[0], Bu.shape[1]))
Dw = np.zeros(shape=(C.shape[0], Bw.shape[1]))

In [7]:
nx = A.shape[0]
nu = Bu.shape[1]
nc = C.shape[0]

eps = 10e-19 # 

gamma = cp.Variable()

X11 = cp.Variable((nx, nx), symmetric=True)
X21 = cp.Variable((nx, nx), symmetric=True)
X32 = cp.Variable((nc, nc), symmetric=True)
P = cp.Variable((nx, nx), symmetric=True)

constrains = []
constrains += [ P >> eps ]

B11 = -P - X11@A - A.T@X11
B12 = X11 - A.T@X21
B13 = -C.T@X32.T
B14 = -X11@Bw

B22 = P + 2*X21
B23 = np.zeros(shape=(nx, nc))
B24 = -X21@Bw

B33 = -np.eye(nc, dtype=float)
B34 = -X32@Dw

B44 = -np.eye(1)*gamma

block = cp.bmat([
    [B11  , B12  , B13  , B14],
    [B12.T, B22  , B23  , B24],
    [B13.T, B23.T, B33  , B34],
    [B14.T, B24.T, B34.T, B44]
])
constrains += [ block << -eps]

prob = cp.Problem(cp.Minimize(None), constraints=constrains)
prob.solve(solver=cp.MOSEK, verbose=True)

                                     CVXPY                                     
                                     v1.5.3                                    
(CVXPY) Jan 06 07:39:24 PM: Your problem has 14 variables, 40 constraints, and 0 parameters.
(CVXPY) Jan 06 07:39:24 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jan 06 07:39:24 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jan 06 07:39:24 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jan 06 07:39:24 PM: Your problem is compiled with the CPP canonicalization backend.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Jan 06 07:39:24 PM: Compiling problem (target solver=MOSEK).
(C

nan

In [8]:
all(np.linalg.eig(P.value).eigenvalues > 0)

True